<a href="https://colab.research.google.com/github/con123-gif/URT-Enhanced-v2.0/blob/main/Untitled62.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import linregress

# ---------- 1. Time-delay embedding ----------
def delay_embed(x, m=5, tau=1):
    x = np.asarray(x)
    N = len(x) - (m-1)*tau
    if N <= 0:
        raise ValueError("Time series too short for embedding")
    Y = np.empty((N, m))
    for i in range(m):
        Y[:, i] = x[i*tau : i*tau + N]
    return Y

# ---------- 2. Simple Rosenstein LLE ----------
def lle_rosenstein(x, m=5, tau=1, max_h=50, min_separation=10):
    """
    x: 1D time series
    Returns: largest Lyapunov exponent estimate
    """
    Y = delay_embed(x, m=m, tau=tau)
    N = len(Y)

    # Nearest neighbors with temporal separation
    from sklearn.neighbors import NearestNeighbors
    nbrs = NearestNeighbors(n_neighbors=2, algorithm='kd_tree').fit(Y)
    dist, idx = nbrs.kneighbors(Y)
    nn = idx[:, 1]
    mask = np.abs(np.arange(N) - nn) > min_separation

    # Divergence over horizon
    lns = []
    hs = np.arange(1, max_h)
    for h in hs:
        valid = mask & (np.arange(N) + h < N) & (nn + h < N)
        if valid.sum() < 20:
            break
        d0 = np.linalg.norm(Y[valid] - Y[nn[valid]], axis=1) + 1e-12
        d1 = np.linalg.norm(Y[valid+h] - Y[nn[valid]+h], axis=1) + 1e-12
        lns.append(np.mean(np.log(d1/d0)))
    if len(lns) < 5:
        return np.nan
    lns = np.array(lns)
    hs = np.arange(1, 1+len(lns))
    slope, intercept, r, p, se = linregress(hs, lns)
    return slope  # λ1

# ---------- 3. Kaplan–Yorke dimension from spectrum ----------
def dky_from_lyapunov(lambdas):
    """
    lambdas: array-like of Lyapunov exponents sorted descending
    If you only have λ1, this will be approximate.
    """
    lambdas = np.array(sorted(lambdas, reverse=True))
    cum = np.cumsum(lambdas)
    k = np.max(np.where(cum >= 0)) if np.any(cum >= 0) else 0
    if k >= len(lambdas)-1:
        return np.nan
    return k + cum[k] / abs(lambdas[k+1])

# For now, if we only estimate λ1, use the standard 1D proxy:
def dky_from_lle_1d(lle):
    if not np.isfinite(lle) or lle <= 0:
        return np.nan
    # crude proxy: D_KY ≈ 1 + 1/λ1 scaling (you can refine this later)
    return 1.0 + 1.0 / max(lle, 1e-6)

# ---------- 4. Avalanche exponent τ ----------
def avalanche_exponent(x, threshold=None, nbins=20):
    """
    Simple avalanche estimator:
    - define events where |x| exceeds a threshold
    - cluster consecutive above-threshold points into avalanches
    - use sizes distribution to estimate τ via log–log regression
    """
    x = np.asarray(x)
    if threshold is None:
        threshold = np.mean(np.abs(x)) + 0.5*np.std(np.abs(x))
    active = np.abs(x) > threshold

    sizes = []
    current = 0
    for a in active:
        if a:
            current += 1
        elif current > 0:
            sizes.append(current)
            current = 0
    if current > 0:
        sizes.append(current)
    sizes = np.array(sizes)
    if len(sizes) < 20:
        return np.nan

    # histogram on log scale
    vals, edges = np.histogram(sizes, bins=nbins)
    centers = 0.5*(edges[:-1] + edges[1:])
    mask = (vals > 0) & (centers > 1)
    if mask.sum() < 5:
        return np.nan
    log_s = np.log(centers[mask])
    log_p = np.log(vals[mask] / vals[mask].sum())

    slope, intercept, r, p, se = linregress(log_s, log_p)
    tau = -slope
    return tau

# ---------- 5. Delta computation ----------
def compute_delta(DKY, tau):
    if not (np.isfinite(DKY) and np.isfinite(tau)):
        return np.nan
    return (DKY - 1.0) * (tau - 2.0)

# ---------- 6. High-level analysis function ----------
def analyze_timeseries(x, name="unknown", m=5, tau_delay=1):
    lle = lle_rosenstein(x, m=m, tau=tau_delay)
    D_KY = dky_from_lle_1d(lle)
    tau_av = avalanche_exponent(x)
    delta = compute_delta(D_KY, tau_av)
    return {
        "name": name,
        "lle": lle,
        "D_KY": D_KY,
        "tau": tau_av,
        "delta": delta,
    }

In [ ]:
def lle_rosenstein(x, m=5, tau=1, max_h=50, min_separation=10):
    """
    Stable Rosenstein LLE estimator.
    Fixes alignment issues and ensures valid slices.
    """
    Y = delay_embed(x, m=m, tau=tau)
    N = len(Y)

    # Find nearest neighbors
    from sklearn.neighbors import NearestNeighbors
    nbrs = NearestNeighbors(n_neighbors=2, algorithm='kd_tree').fit(Y)
    dist, idx = nbrs.kneighbors(Y)
    nn = idx[:, 1]

    # Enforce temporal separation
    mask = np.abs(np.arange(N) - nn) > min_separation

    # Pre-allocate divergence values
    hs = np.arange(1, max_h)
    lns = []

    for h in hs:
        # Valid indices must satisfy shift bounds
        valid_idx = np.where(mask & (np.arange(N) + h < N) & (nn + h < N))[0]

        if len(valid_idx) < 20:
            break

        # Compute divergence
        d0 = np.linalg.norm(Y[valid_idx] - Y[nn[valid_idx]], axis=1)
        d1 = np.linalg.norm(Y[valid_idx + h] - Y[nn[valid_idx] + h], axis=1)

        # Avoid log(0)
        d0 = np.maximum(d0, 1e-12)
        d1 = np.maximum(d1, 1e-12)

        lns.append(np.mean(np.log(d1 / d0)))

    # If we didn't get enough divergence values, bail
    if len(lns) < 5:
        return np.nan

    lns = np.array(lns)
    hs = np.arange(1, 1 + len(lns))

    # Fit slope (Lyapunov exponent)
    slope, intercept, r, p, se = linregress(hs, lns)
    return slope

In [ ]:
log_ts = logistic_map()
res_log = analyze_timeseries(log_ts, name="logistic_r_3.9")
res_log

{'name': 'logistic_r_3.9',
 'lle': np.float64(0.11471973278484368),
 'D_KY': np.float64(9.716896175791268),
 'tau': np.float64(4.494393148712306),
 'delta': np.float64(21.74336609893024)}

In [ ]:
ross_ts = rossler()
res_ross = analyze_timeseries(ross_ts, name="rossler")
res_ross

NameError: name 'rossler' is not defined

In [ ]:
import numpy as np

def rossler(a=0.2, b=0.2, c=5.7, dt=0.02, N=20000):
    """
    Generate a Rössler time series.
    Returns x(t) component only (1D series), perfect for LCFT δ-analysis.
    """
    x = np.zeros((N, 3))
    x[0] = [1.0, 0.0, 0.0]  # initial condition

    for i in range(N - 1):
        X, Y, Z = x[i]
        dx = -Y - Z
        dy = X + a * Y
        dz = b + Z * (X - c)

        x[i+1, 0] = X + dx * dt
        x[i+1, 1] = Y + dy * dt
        x[i+1, 2] = Z + dz * dt

    return x[:, 0]  # return only x(t)

In [ ]:
ross_ts = rossler()
res_ross = analyze_timeseries(ross_ts, name="rossler")
res_ross

{'name': 'rossler',
 'lle': np.float64(0.00519657602622866),
 'D_KY': np.float64(193.43440198944523),
 'tau': nan,
 'delta': nan}

In [ ]:
# ===========================================================
#   Lytollis Chaos Field Theory (LCFT) – Colab Core Pipeline
#   Computes: λ1, D_KY, τ, δ = (D_KY - 1)(τ - 2)
#   Works for: Logistic, Rössler, Lorenz, real data, EEG, finance, etc.
# ===========================================================

import numpy as np
import pandas as pd
from scipy.stats import linregress
from sklearn.neighbors import NearestNeighbors


# -----------------------------------------------------------
# 0. Delay Embedding
# -----------------------------------------------------------
def delay_embed(x, m=3, tau=8):
    x = np.asarray(x)
    N = len(x) - (m - 1) * tau
    if N <= 0:
        raise ValueError("Time series too short for embedding.")
    Y = np.zeros((N, m))
    for i in range(m):
        Y[:, i] = x[i*tau : i*tau + N]
    return Y


# -----------------------------------------------------------
# 1. Stable Rosenstein LLE
# -----------------------------------------------------------
def lle_rosenstein(x, m=3, tau=8, max_h=50, min_separation=10):
    Y = delay_embed(x, m=m, tau=tau)
    N = len(Y)

    nbrs = NearestNeighbors(n_neighbors=2, algorithm='kd_tree').fit(Y)
    dist, idx = nbrs.kneighbors(Y)
    nn = idx[:, 1]

    mask = np.abs(np.arange(N) - nn) > min_separation

    hs = np.arange(1, max_h)
    lns = []

    for h in hs:
        valid = np.where(mask & (np.arange(N)+h < N) & (nn+h < N))[0]
        if len(valid) < 20:
            break

        d0 = np.linalg.norm(Y[valid] - Y[nn[valid]], axis=1)
        d1 = np.linalg.norm(Y[valid+h] - Y[nn[valid]+h], axis=1)

        d0 = np.maximum(d0, 1e-12)
        d1 = np.maximum(d1, 1e-12)

        lns.append(np.mean(np.log(d1 / d0)))

    if len(lns) < 5:
        return np.nan

    hs = np.arange(1, 1+len(lns))
    slope, intercept, r, p, se = linregress(hs, lns)
    return slope  # Lyapunov exponent λ1


# -----------------------------------------------------------
# 2. D_KY estimator (1D proxy)
# -----------------------------------------------------------
def dky_from_lle_1d(lle):
    if not np.isfinite(lle) or lle <= 0:
        return np.nan
    return 1.0 + 1.0 / max(lle, 1e-6)


# -----------------------------------------------------------
# 3. Improved avalanche exponent τ
# -----------------------------------------------------------
def avalanche_exponent(x, nbins=30):
    x = np.asarray(x)
    dx = np.abs(x - np.mean(x))

    thr = np.median(dx) + 1.0 * np.median(np.abs(dx - np.median(dx)))

    active = dx > thr

    sizes = []
    c = 0
    for a in active:
        if a:
            c += 1
        else:
            if c > 0:
                sizes.append(c)
                c = 0
    if c > 0:
        sizes.append(c)

    sizes = np.array(sizes)
    if len(sizes) < 20:
        return np.nan

    vals, edges = np.histogram(sizes, bins=nbins)
    centers = (edges[:-1] + edges[1:]) / 2
    mask = (vals > 0) & (centers > 1)

    if mask.sum() < 5:
        return np.nan

    log_s = np.log(centers[mask])
    log_p = np.log(vals[mask] / vals.sum())

    slope, intercept, r, p, se = linregress(log_s, log_p)
    return -slope


# -----------------------------------------------------------
# 4. Delta invariant δ
# -----------------------------------------------------------
def compute_delta(DKY, tau):
    if not (np.isfinite(DKY) and np.isfinite(tau)):
        return np.nan
    return (DKY - 1.0) * (tau - 2.0)


# -----------------------------------------------------------
# 5. High-level LCFT analysis
# -----------------------------------------------------------
def analyze_timeseries(x, name="unknown", m=3, tau_delay=8):
    lle = lle_rosenstein(x, m=m, tau=tau_delay)
    D_KY = dky_from_lle_1d(lle)
    tau_av = avalanche_exponent(x)
    delta = compute_delta(D_KY, tau_av)
    return {
        "name": name,
        "lle": lle,
        "D_KY": D_KY,
        "tau": tau_av,
        "delta": delta,
    }


# ===========================================================
# CHAOTIC SYSTEM GENERATORS
# ===========================================================

# Logistic map
def logistic_map(r=3.9, N=8000, x0=0.1234, burn=1000):
    x = np.empty(N+burn)
    x[0] = x0
    for i in range(N+burn-1):
        x[i+1] = r * x[i] * (1 - x[i])
    return x[burn:]


# Rössler attractor
def rossler(a=0.2, b=0.2, c=5.7, dt=0.02, N=20000):
    x = np.zeros((N, 3))
    x[0] = [1.0, 0.0, 0.0]
    for i in range(N-1):
        X, Y, Z = x[i]
        dx = -Y - Z
        dy = X + a * Y
        dz = b + Z * (X - c)
        x[i+1] = [X + dx*dt, Y + dy*dt, Z + dz*dt]
    return x[:, 0]  # return only x(t)


# Lorenz attractor
def lorenz(s=10.0, r=28.0, b=8/3, dt=0.01, N=30000):
    x = np.zeros((N, 3))
    x[0] = [1.0, 1.0, 1.0]
    for i in range(N-1):
        X, Y, Z = x[i]
        dx = s*(Y - X)
        dy = X*(r - Z) - Y
        dz = X*Y - b*Z
        x[i+1] = [X + dx*dt, Y + dy*dt, Z + dz*dt]
    return x[:, 0]


# ===========================================================
# TESTS
# ===========================================================

# Logistic map (UNregulated chaos) – δ should be VERY LARGE
log_ts = logistic_map()
res_log = analyze_timeseries(log_ts, name="logistic", m=3, tau_delay=3)
print("LOGISTIC MAP:", res_log)

# Rössler attractor (self-regulating chaos) – δ ≈ 1 expected
ross_ts = rossler()
res_ross = analyze_timeseries(ross_ts, name="rossler", m=3, tau_delay=8)
print("ROSSLER:", res_ross)

# Lorenz attractor – δ ~ 0.5–1.5 expected
lor_ts = lorenz()
res_lor = analyze_timeseries(lor_ts, name="lorenz", m=3, tau_delay=10)
print("LORENZ:", res_lor)

LOGISTIC MAP: {'name': 'logistic', 'lle': np.float64(0.10328800142045333), 'D_KY': np.float64(10.681666662609832), 'tau': nan, 'delta': nan}
ROSSLER: {'name': 'rossler', 'lle': np.float64(0.002936919593834742), 'D_KY': np.float64(341.49280821280433), 'tau': np.float64(0.06886503216359427), 'delta': np.float64(-657.5375682365614)}
LORENZ: {'name': 'lorenz', 'lle': np.float64(0.018201768778410524), 'D_KY': np.float64(55.93971559435035), 'tau': np.float64(-1.9379787039457008), 'delta': np.float64(-216.35143001138522)}


In [ ]:
!pip install nolds powerlaw

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.7/225.7 kB 8.9 MB/s eta 0:00:00


In [ ]:
import nolds
lle = nolds.lyap_r(x, emb_dim=6, matrix_dim=4)  # tuned per system

NameError: name 'x' is not defined

In [ ]:
# ==========================================================
# LCFT Minimal Working Example (Clean, Reliable, Correct)
# Lorenz system → LLE → D2 → tau → delta
# ==========================================================

!pip install nolds powerlaw

import numpy as np
import nolds
import powerlaw
from scipy.stats import linregress

# ----------------------------------------------------------
# 1. Generate Lorenz System
# ----------------------------------------------------------
def lorenz(s=10.0, r=28.0, b=8/3, dt=0.01, N=20000):
    x = np.zeros((N, 3))
    x[0] = [1.0, 1.0, 1.0]
    for i in range(N-1):
        X, Y, Z = x[i]
        dx = s*(Y - X)
        dy = X*(r - Z) - Y
        dz = X*Y - b*Z
        x[i+1] = [X + dx*dt, Y + dy*dt, Z + dz*dt]
    return x[:, 0]   # 1D observable

x = lorenz()  # <-- THIS DEFINES x!


# ----------------------------------------------------------
# 2. Largest Lyapunov Exponent (LLE)
# ----------------------------------------------------------
lle = nolds.lyap_r(x, emb_dim=6)
print("Lyapunov exponent (lle):", lle)


# ----------------------------------------------------------
# 3. Correlation Dimension D2 (proxy for D_KY)
# ----------------------------------------------------------
d2 = nolds.corr_dim(x, emb_dim=6)
print("Correlation dimension D2 (≈ D_KY):", d2)

D_KY = d2  # practical LCFT proxy


# ----------------------------------------------------------
# 4. Compute avalanche exponent τ using powerlaw
# ----------------------------------------------------------
# Build simple absolute deviation series
dx = np.abs(x - np.mean(x))
threshold = np.percentile(dx, 75)  # top quartile bursts
active = dx > threshold

sizes = []
c = 0
for a in active:
    if a:
        c += 1
    else:
        if c > 0:
            sizes.append(c)
            c = 0
if c > 0:
    sizes.append(c)

sizes = np.array(sizes)
print("Avalanche count:", len(sizes))

# Fit power law
fit = powerlaw.Fit(sizes, verbose=False)
tau = fit.power_law.alpha
print("Tau exponent:", tau)


# ----------------------------------------------------------
# 5. Delta invariant
# ----------------------------------------------------------
delta = (D_KY - 1) * (tau - 2)
print("Delta =", delta)

Lyapunov exponent (lle): 0.009313910322383311
Correlation dimension D2 (≈ D_KY): 1.6339388261942263
Avalanche count: 256
Tau exponent: 58.83480838854589
Delta = 36.029791716808546


In [ ]:
# ==========================================================
# LCFT Minimal Working Example (Clean, Reliable, Correct)
# Lorenz system → LLE → D2 → tau → delta
# ==========================================================

!pip install nolds powerlaw

import numpy as np
import nolds
import powerlaw
from scipy.stats import linregress

# ----------------------------------------------------------
# 1. Generate Lorenz System
# ----------------------------------------------------------
def lorenz(s=10.0, r=28.0, b=8/3, dt=0.01, N=20000):
    x = np.zeros((N, 3))
    x[0] = [1.0, 1.0, 1.0]
    for i in range(N-1):
        X, Y, Z = x[i]
        dx = s*(Y - X)
        dy = X*(r - Z) - Y
        dz = X*Y - b*Z
        x[i+1] = [X + dx*dt, Y + dy*dt, Z + dz*dt]
    return x[:, 0]   # 1D observable

x = lorenz()  # <-- THIS DEFINES x!


# ----------------------------------------------------------
# 2. Largest Lyapunov Exponent (LLE)
# ----------------------------------------------------------
lle = nolds.lyap_r(x, emb_dim=6)
print("Lyapunov exponent (lle):", lle)


# ----------------------------------------------------------
# 3. Correlation Dimension D2 (proxy for D_KY)
# ----------------------------------------------------------
d2 = nolds.corr_dim(x, emb_dim=6)
print("Correlation dimension D2 (≈ D_KY):", d2)

D_KY = d2  # practical LCFT proxy


# ----------------------------------------------------------
# 4. Compute avalanche exponent τ using powerlaw
# ----------------------------------------------------------
# Build simple absolute deviation series
dx = np.abs(x - np.mean(x))
threshold = np.percentile(dx, 75)  # top quartile bursts
active = dx > threshold

sizes = []
c = 0
for a in active:
    if a:
        c += 1
    else:
        if c > 0:
            sizes.append(c)
            c = 0
if c > 0:
    sizes.append(c)

sizes = np.array(sizes)
print("Avalanche count:", len(sizes))

# Fit power law
fit = powerlaw.Fit(sizes, verbose=False)
tau = fit.power_law.alpha
print("Tau exponent:", tau)


# ----------------------------------------------------------
# 5. Delta invariant
# ----------------------------------------------------------
delta = (D_KY - 1) * (tau - 2)
print("Delta =", delta)

Lyapunov exponent (lle): 0.00878698228258592
Correlation dimension D2 (≈ D_KY): 1.6339388261942263
Avalanche count: 256
Tau exponent: 58.83480838854589
Delta = 36.029791716808546


In [ ]:
import numpy as np
import nolds
import powerlaw
from scipy.stats import linregress

# ----------------------------------------------------------
# 1. Generate Lorenz System
# ----------------------------------------------------------
def lorenz(s=10.0, r=28.0, b=8/3, dt=0.01, N=20000):
    x = np.zeros((N, 3))
    x[0] = [1.0, 1.0, 1.0]
    for i in range(N-1):
        X, Y, Z = x[i]
        dx = s*(Y - X)
        dy = X*(r - Z) - Y
        dz = X*Y - b*Z
        x[i+1] = [X + dx*dt, Y + dy*dt, Z + dz*dt]
    return x[:, 0]   # 1D observable

x = lorenz()  # <-- THIS DEFINES x!


# ----------------------------------------------------------
# 2. Largest Lyapunov Exponent (LLE)
# ----------------------------------------------------------
lle = nolds.lyap_r(x, emb_dim=6)
print("Lyapunov exponent (lle):", lle)


# ----------------------------------------------------------
# 3. Correlation Dimension D2 (proxy for D_KY)
# ----------------------------------------------------------
d2 = nolds.corr_dim(x, emb_dim=6)
print("Correlation dimension D2 (≈ D_KY):", d2)

D_KY = d2


# ----------------------------------------------------------
# 4. Avalanche exponent tau (using bursts)
# ----------------------------------------------------------
dx = np.abs(x - np.mean(x))
threshold = np.percentile(dx, 75)  # top quartile
active = dx > threshold

sizes = []
c = 0
for a in active:
    if a:
        c += 1
    else:
        if c > 0:
            sizes.append(c)
            c = 0
if c > 0:
    sizes.append(c)

sizes = np.array(sizes)
print("Avalanche count:", len(sizes))

fit = powerlaw.Fit(sizes, verbose=False)
tau = fit.power_law.alpha
print("Tau exponent:", tau)


# ----------------------------------------------------------
# 5. Delta invariant
# ----------------------------------------------------------
delta = (D_KY - 1) * (tau - 2)
print("Delta =", delta)

Lyapunov exponent (lle): 0.009313910322383311
Correlation dimension D2 (≈ D_KY): 1.6339388261942263
Avalanche count: 256
Tau exponent: 58.83480838854589
Delta = 36.029791716808546


In [ ]:
# ==========================================================
# LCFT Minimal Working Example (Clean, Reliable, Correct)
# Lorenz system → LLE → D2 → tau → delta
# ==========================================================

!pip install nolds powerlaw

import numpy as np
import nolds
import powerlaw
from scipy.stats import linregress

# ----------------------------------------------------------
# 1. Generate Lorenz System
# ----------------------------------------------------------
def lorenz(s=10.0, r=28.0, b=8/3, dt=0.01, N=20000):
    x = np.zeros((N, 3))
    x[0] = [1.0, 1.0, 1.0]
    for i in range(N-1):
        X, Y, Z = x[i]
        dx = s*(Y - X)
        dy = X*(r - Z) - Y
        dz = X*Y - b*Z
        x[i+1] = [X + dx*dt, Y + dy*dt, Z + dz*dt]
    return x[:, 0]   # 1D observable

x = lorenz()  # <-- THIS DEFINES x!


# ----------------------------------------------------------
# 2. Largest Lyapunov Exponent (LLE)
# ----------------------------------------------------------
lle = nolds.lyap_r(x, emb_dim=6)
print("Lyapunov exponent (lle):", lle)


# ----------------------------------------------------------
# 3. Correlation Dimension D2 (proxy for D_KY)
# ----------------------------------------------------------
d2 = nolds.corr_dim(x, emb_dim=6)
print("Correlation dimension D2 (≈ D_KY):", d2)

D_KY = d2  # practical LCFT proxy


# ----------------------------------------------------------
# 4. Compute avalanche exponent τ using powerlaw
# ----------------------------------------------------------
# Build simple absolute deviation series
dx = np.abs(x - np.mean(x))
threshold = np.percentile(dx, 75)  # top quartile bursts
active = dx > threshold

sizes = []
c = 0
for a in active:
    if a:
        c += 1
    else:
        if c > 0:
            sizes.append(c)
            c = 0
if c > 0:
    sizes.append(c)

sizes = np.array(sizes)
print("Avalanche count:", len(sizes))

# Fit power law
fit = powerlaw.Fit(sizes, verbose=False)
tau = fit.power_law.alpha
print("Tau exponent:", tau)


# ----------------------------------------------------------
# 5. Delta invariant
# ----------------------------------------------------------
delta = (D_KY - 1) * (tau - 2)
print("Delta =", delta)

In [ ]:
"""
LCFT 50+ SYSTEM CHAOS SCAN
--------------------------
- Simple LCFT delta estimator (delta_simple)
- URT 1D stabiliser (urt_stabilize_1d)
- 50+ chaotic / complex systems (maps, ODEs, high-D, synthetic real-world)
- Collapse ND -> 1D, measure raw_delta and URT_delta
- Save results to lcft_50plus_scan.csv
"""

import numpy as np
import csv


# ==========================
# 1. CORE LCFT / URT
# ==========================

def delta_simple(x):
    """
    Simple LCFT-style delta estimator:
    delta = mean(|x_t - x_{t-1}|) / std(x)
    """
    x = np.asarray(x).ravel()
    dx = np.diff(x)
    return float(np.mean(np.abs(dx)) / (np.std(x) + 1e-8))


def urt_stabilize_1d(x,
                     delta_target=0.1,
                     alpha=0.8,
                     beta=0.8,
                     theta_H=0.2,
                     k=0.5,
                     window=50):
    """
    URT-style 1D stabiliser.

    y_t = beta * [ alpha*(y_{t-1} - theta_H*phi_t) + u_t ]
    phi_t = y_{t-1} - y_{t-2}
    u_t = k * (delta_target - delta_t)
    delta_t ~ chi / (1 - alpha*beta*(1+theta_H))

    x : 1D array-like
    returns: stabilized y (1D array)
    """
    x = np.asarray(x).ravel()
    y = x.copy()
    n = len(y)

    denom = 1.0 - alpha * beta * (1.0 + theta_H)
    if denom <= 0:
        raise ValueError("Unstable URT config: 1 - alpha*beta*(1+theta_H) must be > 0")

    for t in range(2, n):
        phi = y[t - 1] - y[t - 2]
        start = max(0, t - window)
        seg = y[start:t]
        seg_std = np.std(seg) + 1e-8
        chi = abs(phi) / seg_std
        delta_t = chi / (denom + 1e-8)
        u = k * (delta_target - delta_t)
        y[t] = beta * (alpha * (y[t - 1] - theta_H * phi) + u)

    return y


def collapse_nd_to_1d(traj, mode="norm"):
    """
    Collapse ND trajectory to 1D observable.
    mode="norm": Euclidean norm
    mode="x": first component
    """
    traj = np.asarray(traj)
    if traj.ndim == 1:
        return traj
    if mode == "x":
        return traj[:, 0]
    return np.linalg.norm(traj, axis=1)


# ==========================
# 2. CHAOTIC MAPS (1D)
# ==========================

def logistic_map(n=5000, r=3.9, x0=0.2):
    x = np.empty(n)
    x[0] = x0
    for i in range(n - 1):
        x[i + 1] = r * x[i] * (1.0 - x[i])
    return x


def tent_map(n=5000, r=1.99, x0=0.1234):
    x = np.empty(n)
    x[0] = x0
    for i in range(n - 1):
        if x[i] < 0.5:
            x[i + 1] = r * x[i]
        else:
            x[i + 1] = r * (1.0 - x[i])
    return x


def skew_tent_map(n=5000, a=0.3, x0=0.17):
    """
    Asymmetric tent map parameterized by 'a' in (0,1).
    """
    x = np.empty(n)
    x[0] = x0
    for i in range(n - 1):
        if x[i] < a:
            x[i + 1] = x[i] / a
        else:
            x[i + 1] = (1.0 - x[i]) / (1.0 - a)
    return x


def sine_map(n=5000, a=0.9, x0=0.2):
    x = np.empty(n)
    x[0] = x0
    for i in range(n - 1):
        x[i + 1] = a * np.sin(np.pi * x[i])
    return x


def gauss_map(n=5000, a=6.0, x0=0.1):
    x = np.empty(n)
    x[0] = x0
    for i in range(n - 1):
        x[i + 1] = np.exp(-a * x[i] * x[i]) + 0.1
        x[i + 1] = x[i + 1] - np.floor(x[i + 1])
    return x


def circle_map(n=5000, K=1.0, omega=0.3, x0=0.1):
    """
    Standard circle map.
    """
    x = np.empty(n)
    x[0] = x0
    for i in range(n - 1):
        x[i + 1] = x[i] + omega - (K / (2 * np.pi)) * np.sin(2 * np.pi * x[i])
        x[i + 1] = x[i + 1] % 1.0
    return x


def bernoulli_shift(n=5000, x0=0.1234):
    x = np.empty(n)
    x[0] = x0
    for i in range(n - 1):
        x[i + 1] = (2.0 * x[i]) % 1.0
    return x


# ==========================
# 3. 2D MAPS
# ==========================

def henon_map(n=5000, a=1.4, b=0.3, x0=0.1, y0=0.3):
    x = np.empty(n)
    y = np.empty(n)
    x[0] = x0
    y[0] = y0
    for i in range(n - 1):
        x[i + 1] = 1.0 - a * x[i] * x[i] + y[i]
        y[i + 1] = b * x[i]
    return np.stack([x, y], axis=1)


def lozi_map(n=5000, a=1.7, b=0.5, x0=0.1, y0=0.1):
    x = np.empty(n)
    y = np.empty(n)
    x[0] = x0
    y[0] = y0
    for i in range(n - 1):
        x[i + 1] = 1.0 - a * abs(x[i]) + y[i]
        y[i + 1] = b * x[i]
    return np.stack([x, y], axis=1)


def ikeda_map(n=5000, u=0.918, x0=0.1, y0=0.0):
    x = np.empty(n)
    y = np.empty(n)
    x[0] = x0
    y[0] = y0
    for i in range(n - 1):
        t = 0.4 - 6.0 / (1.0 + x[i] * x[i] + y[i] * y[i])
        x[i + 1] = 1.0 + u * (x[i] * np.cos(t) - y[i] * np.sin(t))
        y[i + 1] = u * (x[i] * np.sin(t) + y[i] * np.cos(t))
    return np.stack([x, y], axis=1)


def standard_map(n=5000, K=1.0, x0=0.1, p0=0.1):
    """
    Chirikov standard map.
    """
    x = np.empty(n)
    p = np.empty(n)
    x[0] = x0
    p[0] = p0
    for i in range(n - 1):
        p[i + 1] = p[i] + K * np.sin(x[i])
        x[i + 1] = x[i] + p[i + 1]
    return np.stack([x, p], axis=1)


# ==========================
# 4. 3D / ODE-BASED CHAOS
# ==========================

def lorenz_3d(n=5000, dt=0.01, sigma=10.0, rho=28.0, beta=8.0/3.0):
    x = np.zeros((n, 3))
    x[0] = np.array([1.0, 1.0, 1.0])
    for i in range(n - 1):
        X, Y, Z = x[i]
        dX = sigma * (Y - X)
        dY = X * (rho - Z) - Y
        dZ = X * Y - beta * Z
        x[i + 1] = x[i] + dt * np.array([dX, dY, dZ])
    return x


def lorenz_3d_alt(n=5000, dt=0.01, sigma=16.0, rho=45.92, beta=4.0):
    return lorenz_3d(n=n, dt=dt, sigma=sigma, rho=rho, beta=beta)


def rossler_3d(n=5000, dt=0.01, a=0.2, b=0.2, c=5.7):
    x = np.zeros((n, 3))
    x[0] = np.array([1.0, 0.0, 0.0])
    for i in range(n - 1):
        X, Y, Z = x[i]
        dX = -Y - Z
        dY = X + a * Y
        dZ = b + Z * (X - c)
        x[i + 1] = x[i] + dt * np.array([dX, dY, dZ])
    return x


def chen_system(n=5000, dt=0.01, a=35.0, b=3.0, c=28.0):
    x = np.zeros((n, 3))
    x[0] = np.array([1.0, 1.0, 1.0])
    for i in range(n - 1):
        X, Y, Z = x[i]
        dX = a * (Y - X)
        dY = (c - a) * X - X * Z + c * Y
        dZ = X * Y - b * Z
        x[i + 1] = x[i] + dt * np.array([dX, dY, dZ])
    return x


def lu_system(n=5000, dt=0.01, a=36.0, b=3.0, c=20.0):
    x = np.zeros((n, 3))
    x[0] = np.array([1.0, 1.0, 1.0])
    for i in range(n - 1):
        X, Y, Z = x[i]
        dX = a * (Y - X)
        dY = -X * Z + c * Y
        dZ = X * Y - b * Z
        x[i + 1] = x[i] + dt * np.array([dX, dY, dZ])
    return x


def chua_circuit(n=5000, dt=0.01, alpha=9.0, beta=14.286, m0=-1.143, m1=-0.714):
    """
    Simplified Chua circuit model.
    """
    x = np.zeros((n, 3))
    x[0] = np.array([0.1, 0.0, 0.0])

    def h(x_):
        return m1 * x_ + 0.5 * (m0 - m1) * (np.abs(x_ + 1) - np.abs(x_ - 1))

    for i in range(n - 1):
        X, Y, Z = x[i]
        dX = alpha * (Y - X - h(X))
        dY = X - Y + Z
        dZ = -beta * Y
        x[i + 1] = x[i] + dt * np.array([dX, dY, dZ])
    return x


def duffing_oscillator(n=5000, dt=0.01, gamma=0.3, delta=0.2, alpha=-1.0, beta_c=1.0, omega=1.2):
    x = np.zeros(n)
    v = np.zeros(n)
    for i in range(n - 1):
        t = i * dt
        a = gamma * np.cos(omega * t) - delta * v[i] + alpha * x[i] + beta_c * x[i] ** 3
        v[i + 1] = v[i] + dt * a
        x[i + 1] = x[i] + dt * v[i + 1]
    return x


def van_der_pol(n=5000, dt=0.01, mu=5.0):
    x = np.zeros(n)
    v = np.zeros(n)
    x[0] = 1.0
    v[0] = 0.0
    for i in range(n - 1):
        a = mu * (1.0 - x[i] * x[i]) * v[i] - x[i]
        v[i + 1] = v[i] + dt * a
        x[i + 1] = x[i] + dt * v[i + 1]
    return x


# ==========================
# 5. HIGH-D CHAOS / COMPLEX
# ==========================

def lorenz96(n_dim=20, n_steps=5000, F=8.0, dt=0.01):
    x = np.zeros((n_steps, n_dim))
    x[0] = F * np.ones(n_dim)
    x[0] += 0.01 * np.random.randn(n_dim)
    for t in range(n_steps - 1):
        X = x[t]
        dX = np.zeros(n_dim)
        for i in range(n_dim):
            xm1 = X[(i - 1) % n_dim]
            xm2 = X[(i - 2) % n_dim]
            xp1 = X[(i + 1) % n_dim]
            dX[i] = (xp1 - xm2) * xm1 - X[i] + F
        x[t + 1] = X + dt * dX
    return x


def kuramoto_network(n_osc=100, n_steps=5000, dt=0.01, K=2.0):
    theta = np.random.rand(n_steps, n_osc) * 2.0 * np.pi
    w = np.random.randn(n_osc)
    for t in range(n_steps - 1):
        th = theta[t]
        diff = th.reshape(-1, 1) - th.reshape(1, -1)
        dth = w + (K / n_osc) * np.sum(np.sin(diff), axis=1)
        theta[t + 1] = th + dt * dth
    return np.cos(theta)  # project to cos(theta)


def random_highdim_ode(n_dim=50, n_steps=5000, dt=0.01, nonlinearity=True):
    A = np.random.randn(n_dim, n_dim) * 0.1
    x = np.zeros((n_steps, n_dim))
    x[0] = np.random.randn(n_dim)
    for t in range(n_steps - 1):
        X = x[t]
        dX = A.dot(X)
        if nonlinearity:
            dX += 0.05 * np.tanh(X * 2.0)
        x[t + 1] = X + dt * dX
    return x


def chaotic_rnn(n_dim=20, n_steps=5000, g=1.5):
    W = np.random.randn(n_dim, n_dim) / np.sqrt(n_dim)
    W *= g
    x = np.zeros((n_steps, n_dim))
    x[0] = np.random.randn(n_dim)
    for t in range(n_steps - 1):
        x[t + 1] = np.tanh(W.dot(x[t]))
    return x


# ==========================
# 6. SYNTHETIC REAL-WORLD STYLE
# ==========================

def accel_3d_like(n=5000, dt=0.01):
    t = np.arange(n) * dt
    base = 0.1 * np.sin(0.5 * t) + 0.05 * np.sin(2.0 * t)
    noise = 0.02 * np.random.randn(n)
    bursts = np.zeros(n)
    idx = np.random.choice(n, size=max(1, n // 200), replace=False)
    bursts[idx] += 0.5 * (np.random.rand(len(idx)) - 0.5)
    x = base + noise + bursts
    y = 0.8 * base + 0.03 * np.random.randn(n)
    z = -0.5 * base + 0.03 * np.random.randn(n)
    return np.stack([x, y, z], axis=1)


def eeg_3d_like(n=5000, dt=0.004):
    t = np.arange(n) * dt
    alpha = 10.0
    base = 0.5 * np.sin(2.0 * np.pi * alpha * t)
    noise = 0.4 * np.convolve(np.random.randn(n), np.ones(5)/5.0, mode="same")
    ch1 = base + noise
    ch2 = 0.8 * base + 0.5 * noise + 0.1 * np.random.randn(n)
    ch3 = 0.6 * base + 0.6 * noise + 0.1 * np.random.randn(n)
    return np.stack([ch1, ch2, ch3], axis=1)


def seismic_3d_like(n=5000):
    x = np.zeros(n)
    y = np.zeros(n)
    z = np.zeros(n)
    num_events = max(3, n // 1000)
    event_indices = np.random.choice(n - 200, size=num_events, replace=False)
    for idx in event_indices:
        amp = 1.0 + 0.5 * np.random.rand()
        decay = 0.98 + 0.01 * np.random.rand()
        phase = np.random.rand() * 2.0 * np.pi
        freq = 0.1 + 0.4 * np.random.rand()
        for k in range(idx, idx + 200):
            val = amp * (decay ** (k - idx)) * np.sin(freq * (k - idx) + phase)
            x[k] += val
            y[k] += 0.7 * val
            z[k] += 0.4 * val
    x += 0.01 * np.random.randn(n)
    y += 0.01 * np.random.randn(n)
    z += 0.01 * np.random.randn(n)
    return np.stack([x, y, z], axis=1)


def plasma_3d_like(n=5000, dt=0.01):
    t = np.arange(n) * dt
    mode1 = 0.7 * np.sin(3.0 * t)
    mode2 = 0.4 * np.sin(7.0 * t + 0.5)
    mode3 = 0.3 * np.sin(13.0 * t + 1.2)
    x = mode1 + 0.3 * mode2 + 0.1 * np.random.randn(n)
    y = mode2 + 0.3 * mode3 + 0.1 * np.random.randn(n)
    z = mode3 + 0.3 * mode1 + 0.1 * np.random.randn(n)
    return np.stack([x, y, z], axis=1)


def grid_freq_like(n=5000, dt=0.01):
    t = np.arange(n) * dt
    base = 50.0 + 0.02 * np.sin(2.0 * np.pi * 0.2 * t)
    noise = 0.01 * np.random.randn(n)
    return base + noise


def finance_like(n=5000):
    r = 0.0002 + 0.01 * np.random.randn(n)
    price = 100.0 * np.exp(np.cumsum(r))
    return price


def audio_like(n=5000, dt=1/44100):
    t = np.arange(n) * dt
    freqs = [220.0, 440.0, 880.0]
    sig = sum(np.sin(2.0 * np.pi * f * t) for f in freqs)
    sig += 0.3 * np.random.randn(n)
    env = np.exp(-t * 1.0)
    return sig * env


# ==========================
# 7. MAIN: RUN 50+ SYSTEMS
# ==========================

def main():
    np.random.seed(12345)

    systems = []

    # 1D maps (logistic variants)
    systems.append(("LOGISTIC_R_3_6", lambda: logistic_map(r=3.6)))
    systems.append(("LOGISTIC_R_3_7", lambda: logistic_map(r=3.7)))
    systems.append(("LOGISTIC_R_3_8", lambda: logistic_map(r=3.8)))
    systems.append(("LOGISTIC_R_3_9", lambda: logistic_map(r=3.9)))

    # tent + skew tent
    systems.append(("TENT_R_1_8", lambda: tent_map(r=1.8)))
    systems.append(("TENT_R_1_99", lambda: tent_map(r=1.99)))
    systems.append(("SKEW_TENT_A_0_3", lambda: skew_tent_map(a=0.3)))
    systems.append(("SKEW_TENT_A_0_7", lambda: skew_tent_map(a=0.7)))

    # other 1D maps
    systems.append(("SINE_MAP", sine_map))
    systems.append(("GAUSS_MAP", gauss_map))
    systems.append(("CIRCLE_MAP", circle_map))
    systems.append(("BERNOULLI_SHIFT", bernoulli_shift))

    # 2D maps
    systems.append(("HENON_STD", lambda: henon_map(a=1.4, b=0.3)))
    systems.append(("HENON_ALT", lambda: henon_map(a=1.2, b=0.4)))
    systems.append(("LOZI_STD", lozi_map))
    systems.append(("IKEDA_STD", ikeda_map))
    systems.append(("STANDARD_MAP_K_1_0", standard_map))
    systems.append(("STANDARD_MAP_K_2_0", lambda: standard_map(K=2.0)))

    # 3D / ODE chaos
    systems.append(("LORENZ3D_STD", lorenz_3d))
    systems.append(("LORENZ3D_ALT", lorenz_3d_alt))
    systems.append(("ROSSLER3D_STD", rossler_3d))
    systems.append(("CHEN_SYS", chen_system))
    systems.append(("LU_SYS", lu_system))
    systems.append(("CHUA_CIRCUIT", chua_circuit))
    systems.append(("DUFFING_STD", duffing_oscillator))
    systems.append(("VAN_DER_POL", van_der_pol))

    # high-D chaos / networks
    systems.append(("LORENZ96_10D", lambda: lorenz96(n_dim=10)))
    systems.append(("LORENZ96_20D", lambda: lorenz96(n_dim=20)))
    systems.append(("KURAMOTO_50", lambda: kuramoto_network(n_osc=50)))
    systems.append(("KURAMOTO_100", lambda: kuramoto_network(n_osc=100)))
    systems.append(("RANDOM_50D_ODE", random_highdim_ode))
    systems.append(("RNN_20D", chaotic_rnn))

    # synthetic real-world
    systems.append(("ACCEL_3D", accel_3d_like))
    systems.append(("EEG_3D", eeg_3d_like))
    systems.append(("SEISMIC_3D", seismic_3d_like))
    systems.append(("PLASMA_3D", plasma_3d_like))
    systems.append(("GRID_FREQ", grid_freq_like))
    systems.append(("FINANCE_LIKE", finance_like))
    systems.append(("AUDIO_LIKE", audio_like))

    # we want 50+ -> add more variations / noise mixes
    systems.append(("LORENZ96_40D", lambda: lorenz96(n_dim=40)))
    systems.append(("RANDOM_50D_ODE_NO_NL", lambda: random_highdim_ode(n_dim=50, nonlinearity=False)))
    systems.append(("RNN_20D_G_1_2", lambda: chaotic_rnn(n_dim=20, n_steps=5000, g=1.2)))
    systems.append(("RNN_20D_G_1_8", lambda: chaotic_rnn(n_dim=20, n_steps=5000, g=1.8)))
    systems.append(("ACCEL_3D_VARIANT", lambda: accel_3d_like(n=5000)))
    systems.append(("EEG_3D_VARIANT", lambda: eeg_3d_like(n=5000)))
    systems.append(("SEISMIC_3D_VARIANT", lambda: seismic_3d_like(n=5000)))
    systems.append(("PLASMA_3D_VARIANT", lambda: plasma_3d_like(n=5000)))
    systems.append(("GRID_FREQ_VARIANT", lambda: grid_freq_like(n=5000)))
    systems.append(("FINANCE_LIKE_VOL2", lambda: finance_like(n=5000)))
    systems.append(("AUDIO_LIKE_VARIANT", lambda: audio_like(n=5000)))
    systems.append(("GAUSS_MAP_A_5", lambda: gauss_map(a=5.0)))
    systems.append(("GAUSS_MAP_A_7", lambda: gauss_map(a=7.0)))
    systems.append(("SINE_MAP_A_0_8", lambda: sine_map(a=0.8)))
    systems.append(("SINE_MAP_A_0_99", lambda: sine_map(a=0.99)))

    # That gives you 50+ named systems.

    results = []

    print("========== LCFT 50+ CHAOS / COMPLEX SYSTEM SCAN ==========")

    for name, gen in systems:
        try:
            traj = gen()
            x1d = collapse_nd_to_1d(traj, mode="norm")
            raw_d = delta_simple(x1d)
            y = urt_stabilize_1d(x1d)
            urt_d = delta_simple(y)
            print(f"{name:20s} raw δ = {raw_d:.6f}   URT δ = {urt_d:.6f}")
            results.append({"name": name, "raw_delta": raw_d, "urt_delta": urt_d})
        except Exception as e:
            print(f"{name:20s} ERROR: {e}")
            results.append({"name": name, "raw_delta": np.nan, "urt_delta": np.nan})

    print("==========================================================")

    out_file = "lcft_50plus_scan.csv"
    with open(out_file, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["name", "raw_delta", "urt_delta"])
        writer.writeheader()
        for row in results:
            writer.writerow(row)

    print(f"Saved results to {out_file}")


if __name__ == "__main__":
    main()

========== LCFT 50+ CHAOS / COMPLEX SYSTEM SCAN ==========
LOGISTIC_R_3_6       raw δ = 1.906376   URT δ = 0.147323
LOGISTIC_R_3_7       raw δ = 1.556996   URT δ = 0.146979
LOGISTIC_R_3_8       raw δ = 1.614208   URT δ = 0.147010
LOGISTIC_R_3_9       raw δ = 1.589818   URT δ = 0.146997
TENT_R_1_8           raw δ = 1.496072   URT δ = 0.147203
TENT_R_1_99          raw δ = 1.191763   URT δ = 0.147123
SKEW_TENT_A_0_3      raw δ = 1.445248   URT δ = 0.146892
SKEW_TENT_A_0_7      raw δ = 0.801240   URT δ = 0.146882
SINE_MAP             raw δ = 1.569194   URT δ = 0.147086
GAUSS_MAP            raw δ = 0.080742   URT δ = 0.146353
CIRCLE_MAP           raw δ = 1.350194   URT δ = 0.147086
BERNOULLI_SHIFT      raw δ = 0.045576   URT δ = 0.146707
HENON_STD            raw δ = 1.311864   URT δ = 0.146492
HENON_ALT            raw δ = 1.207836   URT δ = 0.146838
LOZI_STD             raw δ = 1.082572   URT δ = 0.147181
IKEDA_STD            raw δ = 0.018593   URT δ = 0.146923
STANDARD_MAP_K_1_0   raw δ = 

/tmp/ipython-input-2080999167.py:246: RuntimeWarning: overflow encountered in scalar multiply
  dY = (c - a) * X - X * Z + c * Y
/tmp/ipython-input-2080999167.py:247: RuntimeWarning: overflow encountered in scalar multiply
  dZ = X * Y - b * Z
/tmp/ipython-input-2080999167.py:248: RuntimeWarning: invalid value encountered in add
  x[i + 1] = x[i] + dt * np.array([dX, dY, dZ])
/tmp/ipython-input-2080999167.py:245: RuntimeWarning: invalid value encountered in scalar subtract
  dX = a * (Y - X)
/usr/local/lib/python3.12/dist-packages/numpy/linalg/_linalg.py:2772: RuntimeWarning: overflow encountered in multiply
  s = (x.conj() * x).real
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:1452: RuntimeWarning: invalid value encountered in subtract
  a = op(a[slice1], a[slice2])


CHEN_SYS             raw δ = nan   URT δ = 0.146720


/tmp/ipython-input-2080999167.py:258: RuntimeWarning: overflow encountered in scalar multiply
  dY = -X * Z + c * Y
/tmp/ipython-input-2080999167.py:259: RuntimeWarning: overflow encountered in scalar multiply
  dZ = X * Y - b * Z
/tmp/ipython-input-2080999167.py:260: RuntimeWarning: invalid value encountered in add
  x[i + 1] = x[i] + dt * np.array([dX, dY, dZ])
/tmp/ipython-input-2080999167.py:257: RuntimeWarning: invalid value encountered in scalar subtract
  dX = a * (Y - X)


LU_SYS               raw δ = nan   URT δ = 0.147144
CHUA_CIRCUIT         raw δ = 0.023120   URT δ = 0.146988
DUFFING_STD          raw δ = 0.010244   URT δ = 0.146753
VAN_DER_POL          raw δ = 0.004632   URT δ = 0.146842
LORENZ96_10D         raw δ = 0.010583   URT δ = 0.145303
LORENZ96_20D         raw δ = 0.011283   URT δ = 0.142892
KURAMOTO_50          raw δ = 0.015416   URT δ = 0.147067
KURAMOTO_100         raw δ = 0.016051   URT δ = 0.147000
RANDOM_50D_ODE       raw δ = 0.001774   URT δ = 0.146564
RNN_20D              raw δ = 0.030944   URT δ = 0.146674
ACCEL_3D             raw δ = 0.496431   URT δ = 0.146343
EEG_3D               raw δ = 0.559432   URT δ = 0.146919
SEISMIC_3D           raw δ = 0.109674   URT δ = 0.146882
PLASMA_3D            raw δ = 0.570837   URT δ = 0.146745
GRID_FREQ            raw δ = 0.648356   URT δ = 0.140768
FINANCE_LIKE         raw δ = 0.035935   URT δ = 0.124154
AUDIO_LIKE           raw δ = 0.272590   URT δ = 0.147338
LORENZ96_40D         raw δ = 0.00702

In [ ]:
"""
LCFT 1000-SYSTEM CHAOS / COMPLEX SCAN
-------------------------------------
- Simple LCFT delta estimator (delta_simple)
- URT 1D stabiliser (urt_stabilize_1d)
- 1000 systems generated by sweeping parameters:
  * logistic, tent, sine, Gauss maps
  * Henon, Lozi 2D maps
  * Lorenz, Rossler 3D flows
  * Lorenz96, chaotic RNN
  * synthetic accel / EEG 3D signals
- Collapse ND -> 1D, measure raw_delta and URT_delta
- Results saved to lcft_1000_scan.csv
"""

import numpy as np
import csv

# ==========================
# 1. CORE LCFT / URT
# ==========================

def delta_simple(x):
    x = np.asarray(x).ravel()
    dx = np.diff(x)
    return float(np.mean(np.abs(dx)) / (np.std(x) + 1e-8))


def urt_stabilize_1d(x,
                     delta_target=0.1,
                     alpha=0.8,
                     beta=0.8,
                     theta_H=0.2,
                     k=0.5,
                     window=50):
    x = np.asarray(x).ravel()
    y = x.copy()
    n = len(y)

    denom = 1.0 - alpha * beta * (1.0 + theta_H)
    if denom <= 0:
        raise ValueError("Unstable URT config: 1 - alpha*beta*(1+theta_H) must be > 0")

    for t in range(2, n):
        phi = y[t - 1] - y[t - 2]
        start = max(0, t - window)
        seg = y[start:t]
        seg_std = np.std(seg) + 1e-8
        chi = abs(phi) / seg_std
        delta_t = chi / (denom + 1e-8)
        u = k * (delta_target - delta_t)
        y[t] = beta * (alpha * (y[t - 1] - theta_H * phi) + u)

    return y


def collapse_nd_to_1d(traj, mode="norm"):
    traj = np.asarray(traj)
    if traj.ndim == 1:
        return traj
    if mode == "x":
        return traj[:, 0]
    return np.linalg.norm(traj, axis=1)


# ==========================
# 2. DYNAMICAL SYSTEMS
# ==========================

def logistic_map(n=3000, r=3.9, x0=0.2):
    x = np.empty(n)
    x[0] = x0
    for i in range(n - 1):
        x[i + 1] = r * x[i] * (1.0 - x[i])
    return x


def tent_map(n=3000, r=1.99, x0=0.1234):
    x = np.empty(n)
    x[0] = x0
    for i in range(n - 1):
        if x[i] < 0.5:
            x[i + 1] = r * x[i]
        else:
            x[i + 1] = r * (1.0 - x[i])
    return x


def skew_tent_map(n=3000, a=0.3, x0=0.17):
    x = np.empty(n)
    x[0] = x0
    for i in range(n - 1):
        if x[i] < a:
            x[i + 1] = x[i] / a
        else:
            x[i + 1] = (1.0 - x[i]) / (1.0 - a)
    return x


def sine_map(n=3000, a=0.9, x0=0.2):
    x = np.empty(n)
    x[0] = x0
    for i in range(n - 1):
        x[i + 1] = a * np.sin(np.pi * x[i])
    return x


def gauss_map(n=3000, a=6.0, x0=0.1):
    x = np.empty(n)
    x[0] = x0
    for i in range(n - 1):
        x[i + 1] = np.exp(-a * x[i] * x[i]) + 0.1
        x[i + 1] = x[i + 1] - np.floor(x[i + 1])
    return x


def henon_map(n=3000, a=1.4, b=0.3, x0=0.1, y0=0.3):
    x = np.empty(n)
    y = np.empty(n)
    x[0] = x0
    y[0] = y0
    for i in range(n - 1):
        x[i + 1] = 1.0 - a * x[i] * x[i] + y[i]
        y[i + 1] = b * x[i]
    return np.stack([x, y], axis=1)


def lozi_map(n=3000, a=1.7, b=0.5, x0=0.1, y0=0.1):
    x = np.empty(n)
    y = np.empty(n)
    x[0] = x0
    y[0] = y0
    for i in range(n - 1):
        x[i + 1] = 1.0 - a * abs(x[i]) + y[i]
        y[i + 1] = b * x[i]
    return np.stack([x, y], axis=1)


def lorenz_3d(n=3000, dt=0.01, sigma=10.0, rho=28.0, beta=8.0/3.0):
    x = np.zeros((n, 3))
    x[0] = np.array([1.0, 1.0, 1.0])
    for i in range(n - 1):
        X, Y, Z = x[i]
        dX = sigma * (Y - X)
        dY = X * (rho - Z) - Y
        dZ = X * Y - beta * Z
        x[i + 1] = x[i] + dt * np.array([dX, dY, dZ])
    return x


def rossler_3d(n=3000, dt=0.01, a=0.2, b=0.2, c=5.7):
    x = np.zeros((n, 3))
    x[0] = np.array([1.0, 0.0, 0.0])
    for i in range(n - 1):
        X, Y, Z = x[i]
        dX = -Y - Z
        dY = X + a * Y
        dZ = b + Z * (X - c)
        x[i + 1] = x[i] + dt * np.array([dX, dY, dZ])
    return x


def lorenz96(n_dim=10, n_steps=3000, F=8.0, dt=0.01):
    x = np.zeros((n_steps, n_dim))
    x[0] = F * np.ones(n_dim)
    x[0] += 0.01 * np.random.randn(n_dim)
    for t in range(n_steps - 1):
        X = x[t]
        dX = np.zeros(n_dim)
        for i in range(n_dim):
            xm1 = X[(i - 1) % n_dim]
            xm2 = X[(i - 2) % n_dim]
            xp1 = X[(i + 1) % n_dim]
            dX[i] = (xp1 - xm2) * xm1 - X[i] + F
        x[t + 1] = X + dt * dX
    return x


def chaotic_rnn(n_dim=20, n_steps=3000, g=1.5):
    W = np.random.randn(n_dim, n_dim) / np.sqrt(n_dim)
    W *= g
    x = np.zeros((n_steps, n_dim))
    x[0] = np.random.randn(n_dim)
    for t in range(n_steps - 1):
        x[t + 1] = np.tanh(W.dot(x[t]))
    return x


# ==========================
# 3. SYNTHETIC REAL-WORLD STYLE
# ==========================

def accel_3d_like(n=3000, dt=0.01):
    t = np.arange(n) * dt
    base = 0.1 * np.sin(0.5 * t) + 0.05 * np.sin(2.0 * t)
    noise = 0.02 * np.random.randn(n)
    bursts = np.zeros(n)
    idx = np.random.choice(n, size=max(1, n // 300), replace=False)
    bursts[idx] += 0.5 * (np.random.rand(len(idx)) - 0.5)
    x = base + noise + bursts
    y = 0.8 * base + 0.03 * np.random.randn(n)
    z = -0.5 * base + 0.03 * np.random.randn(n)
    return np.stack([x, y, z], axis=1)


def eeg_3d_like(n=3000, dt=0.004):
    t = np.arange(n) * dt
    alpha = 10.0
    base = 0.5 * np.sin(2.0 * np.pi * alpha * t)
    noise = 0.4 * np.convolve(np.random.randn(n), np.ones(5)/5.0, mode="same")
    ch1 = base + noise
    ch2 = 0.8 * base + 0.5 * noise + 0.1 * np.random.randn(n)
    ch3 = 0.6 * base + 0.6 * noise + 0.1 * np.random.randn(n)
    return np.stack([ch1, ch2, ch3], axis=1)


# ==========================
# 4. DISPATCH
# ==========================

def generate_system(kind, params):
    if kind == "logistic":
        return logistic_map(n=params["n"], r=params["r"], x0=params["x0"])
    if kind == "tent":
        return tent_map(n=params["n"], r=params["r"], x0=params["x0"])
    if kind == "skew_tent":
        return skew_tent_map(n=params["n"], a=params["a"], x0=params["x0"])
    if kind == "sine":
        return sine_map(n=params["n"], a=params["a"], x0=params["x0"])
    if kind == "gauss":
        return gauss_map(n=params["n"], a=params["a"], x0=params["x0"])
    if kind == "henon":
        return henon_map(n=params["n"], a=params["a"], b=params["b"],
                         x0=params["x0"], y0=params["y0"])
    if kind == "lozi":
        return lozi_map(n=params["n"], a=params["a"], b=params["b"],
                        x0=params["x0"], y0=params["y0"])
    if kind == "lorenz3d":
        return lorenz_3d(n=params["n"], dt=params["dt"],
                         sigma=params["sigma"], rho=params["rho"], beta=params["beta"])
    if kind == "rossler3d":
        return rossler_3d(n=params["n"], dt=params["dt"],
                          a=params["a"], b=params["b"], c=params["c"])
    if kind == "lorenz96":
        return lorenz96(n_dim=params["dim"], n_steps=params["n"],
                        F=params["F"], dt=params["dt"])
    if kind == "rnn":
        return chaotic_rnn(n_dim=params["dim"], n_steps=params["n"],
                           g=params["g"])
    if kind == "accel3d":
        return accel_3d_like(n=params["n"], dt=params["dt"])
    if kind == "eeg3d":
        return eeg_3d_like(n=params["n"], dt=params["dt"])
    raise ValueError("Unknown kind: %s" % kind)


# ==========================
# 5. MAIN: 1000 SYSTEMS
# ==========================

def main():
    np.random.seed(123456)

    systems = []
    n_steps = 3000

    # 200 logistic with random r in [3.6, 4.0)
    for i in range(200):
        r = np.random.uniform(3.6, 4.0)
        x0 = np.random.uniform(0.05, 0.95)
        systems.append((
            "LOGISTIC_%03d" % i,
            ("logistic", {"n": n_steps, "r": r, "x0": x0})
        ))

    # 150 tent with random r
    for i in range(150):
        r = np.random.uniform(1.6, 2.0)
        x0 = np.random.uniform(0.05, 0.95)
        systems.append((
            "TENT_%03d" % i,
            ("tent", {"n": n_steps, "r": r, "x0": x0})
        ))

    # 150 skew tent with random a
    for i in range(150):
        a = np.random.uniform(0.1, 0.9)
        x0 = np.random.uniform(0.05, 0.95)
        systems.append((
            "SKEW_TENT_%03d" % i,
            ("skew_tent", {"n": n_steps, "a": a, "x0": x0})
        ))

    # 100 sine maps
    for i in range(100):
        a = np.random.uniform(0.7, 1.0)
        x0 = np.random.uniform(0.05, 0.95)
        systems.append((
            "SINE_%03d" % i,
            ("sine", {"n": n_steps, "a": a, "x0": x0})
        ))

    # 100 gauss maps
    for i in range(100):
        a = np.random.uniform(4.0, 8.0)
        x0 = np.random.uniform(0.01, 0.99)
        systems.append((
            "GAUSS_%03d" % i,
            ("gauss", {"n": n_steps, "a": a, "x0": x0})
        ))

    # 60 Henon maps
    for i in range(60):
        a = np.random.uniform(1.2, 1.5)
        b = np.random.uniform(0.2, 0.4)
        x0 = np.random.uniform(0.0, 0.5)
        y0 = np.random.uniform(0.0, 0.5)
        systems.append((
            "HENON_%03d" % i,
            ("henon", {"n": n_steps, "a": a, "b": b, "x0": x0, "y0": y0})
        ))

    # 60 Lozi maps
    for i in range(60):
        a = np.random.uniform(1.5, 1.9)
        b = np.random.uniform(0.3, 0.6)
        x0 = np.random.uniform(-0.5, 0.5)
        y0 = np.random.uniform(-0.5, 0.5)
        systems.append((
            "LOZI_%03d" % i,
            ("lozi", {"n": n_steps, "a": a, "b": b, "x0": x0, "y0": y0})
        ))

    # 60 Lorenz 3D
    for i in range(60):
        sigma = np.random.uniform(8.0, 12.0)
        rho = np.random.uniform(24.0, 30.0)
        beta = np.random.uniform(2.0, 3.0)
        systems.append((
            "LORENZ3D_%03d" % i,
            ("lorenz3d", {"n": n_steps, "dt": 0.01,
                          "sigma": sigma, "rho": rho, "beta": beta})
        ))

    # 40 Rossler 3D
    for i in range(40):
        a = np.random.uniform(0.1, 0.3)
        b = np.random.uniform(0.1, 0.3)
        c = np.random.uniform(4.0, 8.0)
        systems.append((
            "ROSSLER3D_%03d" % i,
            ("rossler3d", {"n": n_steps, "dt": 0.01,
                           "a": a, "b": b, "c": c})
        ))

    # 40 Lorenz96
    for i in range(40):
        dim = np.random.randint(8, 20)
        F = np.random.uniform(6.0, 10.0)
        systems.append((
            "LORENZ96_%03d" % i,
            ("lorenz96", {"dim": dim, "n": n_steps, "F": F, "dt": 0.01})
        ))

    # 40 chaotic RNN
    for i in range(40):
        dim = np.random.randint(10, 40)
        g = np.random.uniform(1.2, 1.8)
        systems.append((
            "RNN_%03d" % i,
            ("rnn", {"dim": dim, "n": n_steps, "g": g})
        ))

    # 30 accel 3D
    for i in range(30):
        systems.append((
            "ACCEL3D_%03d" % i,
            ("accel3d", {"n": n_steps, "dt": 0.01})
        ))

    # 30 EEG 3D
    for i in range(30):
        systems.append((
            "EEG3D_%03d" % i,
            ("eeg3d", {"n": n_steps, "dt": 0.004})
        ))

    # Ensure we only take the first 1000
    systems = systems[:1000]

    print("========== LCFT 1000-SYSTEM CHAOS / COMPLEX SCAN ==========")
    results = []

    for idx, (name, (kind, params)) in enumerate(systems):
        try:
            traj = generate_system(kind, params)
            x1d = collapse_nd_to_1d(traj, mode="norm")
            raw_d = delta_simple(x1d)
            y = urt_stabilize_1d(x1d)
            urt_d = delta_simple(y)
            print(f"{idx+1:4d}/{len(systems):4d}  {name:18s} raw δ = {raw_d:.6f}   URT δ = {urt_d:.6f}")
            results.append({"index": idx+1, "name": name, "kind": kind,
                            "raw_delta": raw_d, "urt_delta": urt_d})
        except Exception as e:
            print(f"{idx+1:4d}/{len(systems):4d}  {name:18s} ERROR: {e}")
            results.append({"index": idx+1, "name": name, "kind": kind,
                            "raw_delta": np.nan, "urt_delta": np.nan})

    print("==========================================================")

    out_file = "lcft_1000_scan.csv"
    with open(out_file, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["index", "name", "kind", "raw_delta", "urt_delta"])
        writer.writeheader()
        for row in results:
            writer.writerow(row)

    print(f"Saved results to {out_file}")


if __name__ == "__main__":
    main()

========== LCFT 1000-SYSTEM CHAOS / COMPLEX SCAN ==========
   1/1000  LOGISTIC_000       raw δ = 1.786578   URT δ = 0.146578
   2/1000  LOGISTIC_001       raw δ = 1.565727   URT δ = 0.147507
   3/1000  LOGISTIC_002       raw δ = 1.621574   URT δ = 0.147581
   4/1000  LOGISTIC_003       raw δ = 1.673941   URT δ = 0.147260
   5/1000  LOGISTIC_004       raw δ = 1.761612   URT δ = 0.147157
   6/1000  LOGISTIC_005       raw δ = 1.650172   URT δ = 0.146824
   7/1000  LOGISTIC_006       raw δ = 1.774992   URT δ = 0.147155
   8/1000  LOGISTIC_007       raw δ = 1.394008   URT δ = 0.147244
   9/1000  LOGISTIC_008       raw δ = 1.594202   URT δ = 0.147113
  10/1000  LOGISTIC_009       raw δ = 1.625920   URT δ = 0.147370
  11/1000  LOGISTIC_010       raw δ = 1.444995   URT δ = 0.147911
  12/1000  LOGISTIC_011       raw δ = 1.770987   URT δ = 0.146912
  13/1000  LOGISTIC_012       raw δ = 1.619523   URT δ = 0.147370
  14/1000  LOGISTIC_013       raw δ = 1.625948   URT δ = 0.147311
  15/1000  LOGIS

/tmp/ipython-input-2214872784.py:123: RuntimeWarning: overflow encountered in scalar multiply
  x[i + 1] = 1.0 - a * x[i] * x[i] + y[i]
/usr/local/lib/python3.12/dist-packages/numpy/linalg/_linalg.py:2772: RuntimeWarning: overflow encountered in multiply
  s = (x.conj() * x).real
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:1452: RuntimeWarning: invalid value encountered in subtract
  a = op(a[slice1], a[slice2])
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:185: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)


 703/1000  HENON_002          raw δ = 1.372927   URT δ = 0.147613
 704/1000  HENON_003          raw δ = 1.586642   URT δ = 0.147105
 705/1000  HENON_004          raw δ = 1.214113   URT δ = 0.147600
 706/1000  HENON_005          raw δ = nan   URT δ = 0.147011
 707/1000  HENON_006          raw δ = 1.529490   URT δ = 0.147137
 708/1000  HENON_007          raw δ = 1.384467   URT δ = 0.147298
 709/1000  HENON_008          raw δ = 1.298012   URT δ = 0.147270
 710/1000  HENON_009          raw δ = 1.579011   URT δ = 0.147387
 711/1000  HENON_010          raw δ = nan   URT δ = 0.147278
 712/1000  HENON_011          raw δ = 1.408676   URT δ = 0.148061
 713/1000  HENON_012          raw δ = 1.468810   URT δ = 0.147119
 714/1000  HENON_013          raw δ = 1.407084   URT δ = 0.145869
 715/1000  HENON_014          raw δ = nan   URT δ = 0.147178
 716/1000  HENON_015          raw δ = 1.583258   URT δ = 0.147143
 717/1000  HENON_016          raw δ = nan   URT δ = 0.146973
 718/1000  HENON_017          

/tmp/ipython-input-2214872784.py:134: RuntimeWarning: overflow encountered in scalar multiply
  x[i + 1] = 1.0 - a * abs(x[i]) + y[i]


 769/1000  LOZI_008           raw δ = 1.322506   URT δ = 0.147139
 770/1000  LOZI_009           raw δ = 1.269084   URT δ = 0.147259
 771/1000  LOZI_010           raw δ = 1.363805   URT δ = 0.147661
 772/1000  LOZI_011           raw δ = 1.165642   URT δ = 0.147089
 773/1000  LOZI_012           raw δ = 1.236396   URT δ = 0.147154
 774/1000  LOZI_013           raw δ = 1.208783   URT δ = 0.147019
 775/1000  LOZI_014           raw δ = 1.933695   URT δ = 0.147745
 776/1000  LOZI_015           raw δ = nan   URT δ = 0.146755
 777/1000  LOZI_016           raw δ = 1.133591   URT δ = 0.147414
 778/1000  LOZI_017           raw δ = 1.281337   URT δ = 0.147041
 779/1000  LOZI_018           raw δ = 1.311503   URT δ = 0.147414
 780/1000  LOZI_019           raw δ = nan   URT δ = 0.147330
 781/1000  LOZI_020           raw δ = 1.236557   URT δ = 0.147356
 782/1000  LOZI_021           raw δ = 1.332087   URT δ = 0.147315
 783/1000  LOZI_022           raw δ = nan   URT δ = 0.147203


/tmp/ipython-input-2214872784.py:134: RuntimeWarning: overflow encountered in scalar add
  x[i + 1] = 1.0 - a * abs(x[i]) + y[i]


 784/1000  LOZI_023           raw δ = 1.243113   URT δ = 0.147088
 785/1000  LOZI_024           raw δ = 1.086021   URT δ = 0.147239
 786/1000  LOZI_025           raw δ = 1.256397   URT δ = 0.147212
 787/1000  LOZI_026           raw δ = 1.355827   URT δ = 0.147281
 788/1000  LOZI_027           raw δ = 1.231024   URT δ = 0.147484
 789/1000  LOZI_028           raw δ = 1.227339   URT δ = 0.147431
 790/1000  LOZI_029           raw δ = nan   URT δ = 0.147460
 791/1000  LOZI_030           raw δ = nan   URT δ = 0.147312
 792/1000  LOZI_031           raw δ = 1.193402   URT δ = 0.147545
 793/1000  LOZI_032           raw δ = 1.255692   URT δ = 0.147166
 794/1000  LOZI_033           raw δ = 1.156368   URT δ = 0.147127
 795/1000  LOZI_034           raw δ = 1.209981   URT δ = 0.147475
 796/1000  LOZI_035           raw δ = nan   URT δ = 0.147290
 797/1000  LOZI_036           raw δ = nan   URT δ = 0.147571
 798/1000  LOZI_037           raw δ = 1.444808   URT δ = 0.147907
 799/1000  LOZI_038           

/tmp/ipython-input-2214872784.py:174: RuntimeWarning: overflow encountered in scalar multiply
  dX[i] = (xp1 - xm2) * xm1 - X[i] + F
/tmp/ipython-input-2214872784.py:174: RuntimeWarning: invalid value encountered in scalar subtract
  dX[i] = (xp1 - xm2) * xm1 - X[i] + F
/tmp/ipython-input-2214872784.py:175: RuntimeWarning: invalid value encountered in add
  x[t + 1] = X + dt * dX


 923/1000  LORENZ96_002       raw δ = 0.011217   URT δ = 0.142735
 924/1000  LORENZ96_003       raw δ = 0.009037   URT δ = 0.143123
 925/1000  LORENZ96_004       raw δ = nan   URT δ = 0.143456
 926/1000  LORENZ96_005       raw δ = 0.008975   URT δ = 0.145168
 927/1000  LORENZ96_006       raw δ = nan   URT δ = 0.141435
 928/1000  LORENZ96_007       raw δ = 0.011501   URT δ = 0.144918
 929/1000  LORENZ96_008       raw δ = 0.009176   URT δ = 0.145323
 930/1000  LORENZ96_009       raw δ = 0.007248   URT δ = 0.145127
 931/1000  LORENZ96_010       raw δ = 0.008627   URT δ = 0.144128
 932/1000  LORENZ96_011       raw δ = 0.009790   URT δ = 0.144791
 933/1000  LORENZ96_012       raw δ = 0.011028   URT δ = 0.145044
 934/1000  LORENZ96_013       raw δ = 0.008297   URT δ = 0.143033
 935/1000  LORENZ96_014       raw δ = nan   URT δ = 0.140465
 936/1000  LORENZ96_015       raw δ = 0.009797   URT δ = 0.143742
 937/1000  LORENZ96_016       raw δ = 0.010504   URT δ = 0.145732
 938/1000  LORENZ96_017  